In [2]:
import pandas as pd

# path to your file
file_path = "cicids2018\cic.csv"

# load dataframe
df = pd.read_csv(file_path)

# check shape (rows, columns)
print("Shape:", df.shape)

# optional: preview
print(df.head())

Shape: (1048575, 80)
   Dst Port  Protocol            Timestamp  Flow Duration  Tot Fwd Pkts  \
0         0         0  14/02/2018 08:31:01      112641719             3   
1         0         0  14/02/2018 08:33:50      112641466             3   
2         0         0  14/02/2018 08:36:39      112638623             3   
3        22         6  14/02/2018 08:40:13        6453966            15   
4        22         6  14/02/2018 08:40:23        8804066            14   

   Tot Bwd Pkts  TotLen Fwd Pkts  TotLen Bwd Pkts  Fwd Pkt Len Max  \
0             0                0                0                0   
1             0                0                0                0   
2             0                0                0                0   
3            10             1239             2273              744   
4            11             1143             2209              744   

   Fwd Pkt Len Min  ...  Fwd Seg Size Min  Active Mean  Active Std  \
0                0  ...              

In [3]:
import pandas as pd
import glob
import os

# 📂 folder containing all CICIDS CSV files
input_folder = "CIC-IDS-2018-Dataset/*.csv"

# 📍 output file path
output_file = "cicids2018/combined_cicids2018.csv"

# get all csv files
files = glob.glob(input_folder)

print(f"Found {len(files)} files")

df_list = []

for file in files:
    print(f"Reading: {file}")
    
    temp_df = pd.read_csv(file, low_memory=False)
    
    df_list.append(temp_df)

# combine all
combined_df = pd.concat(df_list, ignore_index=True)

# save to CSV
combined_df.to_csv(output_file, index=False)

print("✅ Combined CSV saved at:", os.path.abspath(output_file))
print("Shape:", combined_df.shape)

Found 10 files
Reading: CIC-IDS-2018-Dataset\Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
Reading: CIC-IDS-2018-Dataset\Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
✅ Combined CSV saved at: d:\Python\GeneratedLabelledFlows\cicids2018\combined_cicids2018.csv
Shape: (16233002, 84)


In [7]:
cols_to_drop = [
    "Flow ID",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Total Fwd Packets", "Total Backward Packets",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Total Length of Fwd Packets", "Subflow Bwd Bytes",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "RST Flag Count"
]

combined_df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("After column drop:", df.shape)

MemoryError: Unable to allocate 248. MiB for an array with shape (2, 16233002) and data type object

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cicids2018\combined_cicids2018.csv", low_memory=False)

print("Initial Shape:", df.shape)

MemoryError: Unable to allocate 124. MiB for an array with shape (16233002,) and data type int64

In [2]:
import pandas as pd
import numpy as np

input_file = "cicids2018/combined_cicids2018.csv"
output_file = "cleaned_cicids2018.csv"

chunksize = 100000  # adjust (50k if RAM is low)

# columns to drop
cols_to_drop = [
    "Flow ID",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Total Fwd Packets", "Total Backward Packets",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Total Length of Fwd Packets", "Subflow Bwd Bytes",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "RST Flag Count"
]

allowed_classes = ["BENIGN", "DDoS", "DoS Hulk", "PortScan"]

first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=False):
    
    print("Processing chunk...")
    
    # drop columns
    chunk.drop(columns=[c for c in cols_to_drop if c in chunk.columns], inplace=True)
    
    # remove duplicates (within chunk)
    chunk.drop_duplicates(inplace=True)
    
    # remove invalid rows
    chunk = chunk[chunk['Label'].notna()]
    chunk = chunk[chunk['Source IP'].notna()]
    chunk = chunk[chunk['Destination IP'].notna()]
    
    # timestamp fix
    chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
    chunk = chunk[chunk['Timestamp'].notna()]
    
    # remove inf/nan
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    chunk.dropna(inplace=True)
    
    # class filtering
    chunk = chunk[chunk['Label'].isin(allowed_classes)]
    
    # drop non-model features
    chunk.drop(columns=["Source IP", "Destination IP", "Timestamp"], 
               inplace=True, errors='ignore')
    
    # save
    if first_chunk:
        chunk.to_csv(output_file, index=False)
        first_chunk = False
    else:
        chunk.to_csv(output_file, mode='a', header=False, index=False)

print("✅ Done! Cleaned dataset saved.")

Processing chunk...


KeyError: 'Source IP'

In [ ]:
import pandas as pd
import numpy as np

input_file = "cicids2018/combined_cicids2018.csv"
output_file = "cleaned_cicids2018.csv"

chunksize = 100000

cols_to_drop = [
    "Flow ID",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Total Fwd Packets", "Total Backward Packets",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Total Length of Fwd Packets", "Subflow Bwd Bytes",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "RST Flag Count"
]

allowed_classes = ["BENIGN", "DDoS", "DoS Hulk", "PortScan"]

first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=False):
    
    print("Processing chunk...")
    
    # ✅ FIX: remove spaces in column names
    chunk.columns = chunk.columns.str.strip()
    
    # drop columns safely
    chunk.drop(columns=[c for c in cols_to_drop if c in chunk.columns], inplace=True)
    
    # remove duplicates
    chunk.drop_duplicates(inplace=True)
    
    # safe column checks
    if 'Label' in chunk.columns:
        chunk = chunk[chunk['Label'].notna()]
    
    if 'Source IP' in chunk.columns:
        chunk = chunk[chunk['Source IP'].notna()]
        
    if 'Destination IP' in chunk.columns:
        chunk = chunk[chunk['Destination IP'].notna()]
    
    # timestamp
    if 'Timestamp' in chunk.columns:
        chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
        chunk = chunk[chunk['Timestamp'].notna()]
    
    # remove inf/nan
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    chunk.dropna(inplace=True)
    
    # class filtering
    if 'Label' in chunk.columns:
        chunk = chunk[chunk['Label'].isin(allowed_classes)]
    
    # drop non-model features
    chunk.drop(columns=["Source IP", "Destination IP", "Timestamp"], 
               inplace=True, errors='ignore')
    
    # save
    if first_chunk:
        chunk.to_csv(output_file, index=False)
        first_chunk = False
    else:
        chunk.to_csv(output_file, mode='a', header=False, index=False)

print("✅ Done! No more KeyError.")

In [4]:
import pandas as pd

df = pd.read_csv("cleaned_cicids2018.csv")

print(df.shape)

(0, 78)


In [ ]:
import pandas as pd
import numpy as np

input_file = "cicids2018/combined_cicids2018.csv"
output_file = "cleaned_cicids2018.csv"

chunksize = 100000

cols_to_drop = [
    "Flow ID",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Total Fwd Packets", "Total Backward Packets",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Total Length of Fwd Packets", "Subflow Bwd Bytes",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "RST Flag Count"
]

first_chunk = True

for i, chunk in enumerate(pd.read_csv(input_file, chunksize=chunksize, low_memory=False)):
    
    print(f"\n🔹 Processing chunk {i+1}")
    print("Initial:", chunk.shape)

    # ✅ FIX 1: clean column names
    chunk.columns = chunk.columns.str.strip()

    # ✅ FIX 2: normalize labels (THIS is where your fix goes)
    if 'Label' in chunk.columns:
        chunk['Label'] = chunk['Label'].astype(str).str.strip().str.upper()

    # drop unwanted columns
    chunk.drop(columns=[c for c in cols_to_drop if c in chunk.columns], inplace=True)
    print("After column drop:", chunk.shape)

    # remove duplicates
    chunk.drop_duplicates(inplace=True)
    print("After duplicates:", chunk.shape)

    # basic filtering
    if 'Label' in chunk.columns:
        chunk = chunk[chunk['Label'].notna()]

    if 'Source IP' in chunk.columns:
        chunk = chunk[chunk['Source IP'].notna()]

    if 'Destination IP' in chunk.columns:
        chunk = chunk[chunk['Destination IP'].notna()]

    print("After basic filtering:", chunk.shape)

    # timestamp cleaning
    if 'Timestamp' in chunk.columns:
        chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
        chunk = chunk[chunk['Timestamp'].notna()]

    print("After timestamp:", chunk.shape)

    # replace inf values
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)

    # ⚠️ SAFE NaN handling (do NOT drop everything)
    if 'Label' in chunk.columns:
        chunk.dropna(subset=['Label'], inplace=True)

    print("After NaN handling:", chunk.shape)

    # DEBUG (only first chunk)
    if i == 0:
        print("Sample labels AFTER FIX:\n", chunk['Label'].value_counts().head())

    # ❌ DO NOT filter classes here (important!)
    # we will do it later safely

    # remove non-model features
    chunk.drop(columns=["Source IP", "Destination IP", "Timestamp"],
               inplace=True, errors='ignore')

    print("Final chunk shape:", chunk.shape)

    # save chunk
    if len(chunk) > 0:
        if first_chunk:
            chunk.to_csv(output_file, index=False)
            first_chunk = False
        else:
            chunk.to_csv(output_file, mode='a', header=False, index=False)

print("\n✅ DONE: Cleaned dataset saved successfully!")


🔹 Processing chunk 1
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (99538, 79)
After basic filtering: (99538, 79)
After timestamp: (99538, 79)
After NaN handling: (99538, 79)
Sample labels AFTER FIX:
 Label
BOT       79720
BENIGN    19818
Name: count, dtype: int64
Final chunk shape: (99538, 78)

🔹 Processing chunk 2
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (99737, 79)
After basic filtering: (99737, 79)
After timestamp: (99737, 79)
After NaN handling: (99737, 79)
Final chunk shape: (99737, 78)

🔹 Processing chunk 3
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (99775, 79)
After basic filtering: (99775, 79)
After timestamp: (99775, 79)
After NaN handling: (99775, 79)
Final chunk shape: (99775, 78)

🔹 Processing chunk 4
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (99891, 79)
After basic filtering: (99891, 79)
After timestamp: (99891, 79)
After NaN handling: (99891, 79)
Final chu

C:\Users\Admin\AppData\Local\Temp\ipykernel_2608\3001634761.py:59: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')



🔹 Processing chunk 13
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (96770, 79)
After basic filtering: (96770, 79)
After timestamp: (96770, 79)
After NaN handling: (96770, 79)
Final chunk shape: (96770, 78)


C:\Users\Admin\AppData\Local\Temp\ipykernel_2608\3001634761.py:59: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')



🔹 Processing chunk 14
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (97716, 79)
After basic filtering: (97716, 79)
After timestamp: (97716, 79)
After NaN handling: (97716, 79)
Final chunk shape: (97716, 78)


...

After NaN handling: (100000, 79)
Final chunk shape: (100000, 78)

🔹 Processing chunk 68
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (100000, 79)
After basic filtering: (100000, 79)


C:\Users\Admin\AppData\Local\Temp\ipykernel_2608\3001634761.py:59: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')


After timestamp: (100000, 79)
After NaN handling: (100000, 79)
Final chunk shape: (100000, 78)

🔹 Processing chunk 69
Initial: (100000, 84)
After column drop: (100000, 79)
After duplicates: (100000, 79)
After basic filtering: (100000, 79)
After timestamp: (100000, 79)


C:\Users\Admin\AppData\Local\Temp\ipykernel_2608\3001634761.py:59: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')


After NaN handling: (100000, 79)
Final chunk shape: (100000, 78)


In [1]:
import pandas as pd

df = pd.read_csv("cleaned_cicids2018.csv")

# labels already uppercased, but safe:
df['Label'] = df['Label'].str.upper()

allowed_classes = ["BENIGN", "DDOS", "DOS HULK", "PORTSCAN"]

df = df[df['Label'].isin(allowed_classes)]

print(df.shape)
print(df['Label'].value_counts())

C:\Users\Admin\AppData\Local\Temp\ipykernel_8640\3596702449.py:3: DtypeWarning: Columns (75,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_cicids2018.csv")


(5404402, 78)
Label
BENIGN    5404402
Name: count, dtype: int64


In [2]:
print(df.shape)
print(df['Label'].value_counts())

(5404402, 78)
Label
BENIGN    5404402
Name: count, dtype: int64


In [3]:
chunk = next(pd.read_csv("cicids2018/combined_cicids2018.csv", chunksize=200000))

chunk.columns = chunk.columns.str.strip()
chunk['Label'] = chunk['Label'].astype(str).str.strip().str.upper()

print(chunk['Label'].value_counts())

Label
BOT       162906
BENIGN     37094
Name: count, dtype: int64


In [5]:
import pandas as pd

input_file = "cicids2018/combined_cicids2018.csv"
output_file = "cleaned_cicids2018.csv"

chunksize = 100000

# columns to remove
cols_to_drop = [
    "Flow ID",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Total Fwd Packets", "Total Backward Packets",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Total Length of Fwd Packets", "Subflow Bwd Bytes",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "RST Flag Count"
]

# only these labels
allowed_classes = ["BENIGN", "DDOS", "DOS HULK", "PORTSCAN"]

first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=False):

    # clean column names
    chunk.columns = chunk.columns.str.strip()

    # clean labels
    chunk['Label'] = chunk['Label'].astype(str).str.strip().str.upper()

    # drop columns
    chunk.drop(columns=[c for c in cols_to_drop if c in chunk.columns], inplace=True)

    # drop null rows (simple)
    chunk.dropna(inplace=True)

    # keep only required classes
    chunk = chunk[chunk['Label'].isin(allowed_classes)]

    # save
    if len(chunk) > 0:
        if first_chunk:
            chunk.to_csv(output_file, index=False)
            first_chunk = False
        else:
            chunk.to_csv(output_file, mode='a', header=False, index=False)

print("✅ Done. Clean dataset saved.")

✅ Done. Clean dataset saved.
